# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(["git", "clone", clone_url, str(repo_dir)], check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")

In [ ]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [ ]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

In [ ]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [ ]:
run_pytest("tests/test_bpe.py")

In [ ]:
# BPE 구현 후 작은 말뭉치로 인코딩/디코딩 복원을 확인합니다.
try:
    from bpe import BPETokenizer

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)

## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [ ]:
run_pytest("tests/test_dataset.py")

In [ ]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=300, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)

## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [ ]:
run_pytest("tests/test_attention.py")

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [ ]:
run_pytest("tests/test_model.py")

In [ ]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)

## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [ ]:
run_pytest("tests/test_train.py")

In [ ]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)

In [ ]:
# 본 사전 학습: Light/Basic 설정으로 실제 checkpoint와 학습 기록을 저장합니다.
# 시간이 부족하면 PRETRAIN_PRESET을 "Light"로 바꾸고, 제출 기본 실험은 "Basic"을 권장합니다.
from pathlib import Path
import json
import time
import traceback
import torch

from bpe import BPETokenizer
from dataset import create_dataloader
from model import GPTModel
from train import train_model, save_checkpoint

PRETRAIN_PRESET = "Light"  # "Light" 또는 "Basic"
FORCE_RETRAIN_BPE = False  # 이미 저장된 vocab을 무시하고 다시 학습하려면 True

presets = {
    "Light": {
        "corpus_size": 500_000,
        "vocab_size": 2000,
        "context_length": 64,
        "batch_size": 8,
        "emb_dim": 128,
        "n_heads": 4,
        "n_layers": 2,
        "num_epochs": 2,
        "eval_freq": 100,
        "eval_iter": 10,
        "ckpt_freq": 500,
        "learning_rate": 3e-4,
    },
    "Basic": {
        "corpus_size": 1_500_000,
        "vocab_size": 3000,
        "context_length": 128,
        "batch_size": 8,
        "emb_dim": 128,
        "n_heads": 4,
        "n_layers": 2,
        "num_epochs": 3,
        "eval_freq": 100,
        "eval_iter": 10,
        "ckpt_freq": 500,
        "learning_rate": 3e-4,
    },
}

settings = presets[PRETRAIN_PRESET]
output_dir = Path("outputs") / f"pretrain_{PRETRAIN_PRESET.lower()}"
ckpt_dir = output_dir / "checkpoints"
output_dir.mkdir(parents=True, exist_ok=True)
ckpt_dir.mkdir(parents=True, exist_ok=True)
status_path = output_dir / "status.json"


def write_status(stage, **extra):
    status = {
        "preset": PRETRAIN_PRESET,
        "stage": stage,
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        **extra,
    }
    with status_path.open("w", encoding="utf-8") as f:
        json.dump(status, f, ensure_ascii=False, indent=2)
    print("[stage]", stage, extra, flush=True)

try:
    write_status("start")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("preset:", PRETRAIN_PRESET, flush=True)
    print("device:", device, flush=True)

    train_text_path = Path("data/nsmc_lm_train.txt")
    val_text_path = Path("data/nsmc_lm_val.txt")
    write_status("loading_text")
    train_text = train_text_path.read_text(encoding="utf-8")
    val_text = val_text_path.read_text(encoding="utf-8")

    vocab_size = settings["vocab_size"]
    corpus_size = settings["corpus_size"]
    vocab_path = Path(f"data/nsmc_bpe_vocab_{vocab_size}.json")

    if vocab_size <= 260:
        raise ValueError("byte-level BPE는 특수 토큰 4개 + byte 토큰 256개가 기본이라 vocab_size는 260보다 커야 합니다.")
    if corpus_size <= 0 or len(train_text[:corpus_size]) == 0:
        raise ValueError("BPE 학습에 사용할 corpus가 비어 있습니다. 데이터 준비 셀과 corpus_size를 확인하세요.")

    tokenizer = BPETokenizer(vocab_size=vocab_size)
    should_load_vocab = vocab_path.exists() and not FORCE_RETRAIN_BPE
    bpe_info = {
        "vocab_size": vocab_size,
        "corpus_size": corpus_size,
        "corpus_slice_length": len(train_text[:corpus_size]),
        "vocab_path": str(vocab_path),
        "bpe_action": "load" if should_load_vocab else "train",
        "loaded_existing_vocab": should_load_vocab,
        "force_retrain_bpe": FORCE_RETRAIN_BPE,
        "training_time_sec": None,
        "load_time_sec": None,
    }

    if should_load_vocab:
        write_status("loading_bpe_vocab", vocab_path=str(vocab_path))
        load_start = time.time()
        tokenizer.load(vocab_path)
        bpe_info["load_time_sec"] = time.time() - load_start
    else:
        write_status(
            "training_bpe_vocab",
            vocab_path=str(vocab_path),
            note="이 단계가 가장 오래 걸릴 수 있습니다. 이미 vocab 파일이 있으면 FORCE_RETRAIN_BPE=False일 때는 학습하지 않고 load합니다.",
        )
        bpe_start = time.time()
        tokenizer.train(train_text[:corpus_size])
        bpe_info["training_time_sec"] = time.time() - bpe_start
        tokenizer.save(vocab_path)

    bpe_info["actual_vocab_size"] = len(tokenizer.id_to_token)
    bpe_info["merge_count"] = len(tokenizer.merges)
    if bpe_info["actual_vocab_size"] <= 260:
        raise ValueError("BPE merge가 하나도 만들어지지 않았습니다. vocab_size와 corpus 내용을 확인하세요.")

    with open(output_dir / "bpe_info.json", "w", encoding="utf-8") as f:
        json.dump(bpe_info, f, ensure_ascii=False, indent=2)

    write_status("encoding_text")
    encode_start = time.time()
    train_ids = tokenizer.encode(train_text[:corpus_size])
    val_ids = tokenizer.encode(val_text[: min(200_000, len(val_text))])
    encode_time = time.time() - encode_start

    config = {
        "vocab_size": vocab_size,
        "context_length": settings["context_length"],
        "emb_dim": settings["emb_dim"],
        "n_heads": settings["n_heads"],
        "n_layers": settings["n_layers"],
        "drop_rate": 0.1,
        "qkv_bias": False,
    }

    write_status("creating_dataloaders", train_tokens=len(train_ids), val_tokens=len(val_ids))
    train_loader = create_dataloader(
        train_ids,
        context_length=config["context_length"],
        batch_size=settings["batch_size"],
        stride=config["context_length"],
        shuffle=True,
        drop_last=True,
    )
    val_loader = create_dataloader(
        val_ids,
        context_length=config["context_length"],
        batch_size=settings["batch_size"],
        stride=config["context_length"],
        shuffle=False,
        drop_last=False,
    )

    write_status("creating_model")
    model = GPTModel(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings["learning_rate"])
    param_count = sum(p.numel() for p in model.parameters())

    run_config = {
        "preset": PRETRAIN_PRESET,
        "settings": settings,
        "model_config": config,
        "parameter_count": param_count,
        "train_token_count": len(train_ids),
        "val_token_count": len(val_ids),
        "train_batches": len(train_loader),
        "val_batches": len(val_loader),
        "encode_time_sec": encode_time,
        "device": str(device),
    }
    with open(output_dir / "run_config.json", "w", encoding="utf-8") as f:
        json.dump(run_config, f, ensure_ascii=False, indent=2)

    write_status("training_model", train_batches=len(train_loader), val_batches=len(val_loader))
    train_start = time.time()
    pretrain_history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        device=device,
        num_epochs=settings["num_epochs"],
        eval_freq=settings["eval_freq"],
        eval_iter=settings["eval_iter"],
        start_context="이 영화는",
        tokenizer=tokenizer,
        ckpt_freq=settings["ckpt_freq"],
        ckpt_dir=ckpt_dir,
    )
    pretrain_history["total_training_time_sec"] = time.time() - train_start

    final_ckpt_path = output_dir / "final_checkpoint.pt"
    final_step = pretrain_history["global_step"][-1] if pretrain_history["global_step"] else 0
    save_checkpoint(
        model=model,
        optimizer=optimizer,
        epoch=settings["num_epochs"] - 1,
        global_step=final_step,
        path=str(final_ckpt_path),
    )
    pretrain_history["final_checkpoint_path"] = str(final_ckpt_path)

    with open(output_dir / "pretrain_history.json", "w", encoding="utf-8") as f:
        json.dump(pretrain_history, f, ensure_ascii=False, indent=2)

    write_status("done", final_checkpoint_path=str(final_ckpt_path))
    print("run_config:", run_config, flush=True)
    print("history:", pretrain_history, flush=True)
    print("saved to:", output_dir, flush=True)
except Exception as e:
    error_text = traceback.format_exc()
    write_status("error", error=repr(e), traceback=error_text)
    print(error_text)
    raise


In [ ]:
# 본 사전 학습 결과 그래프: train/validation loss와 epoch별 학습 시간을 확인합니다.
from pathlib import Path
import json
import matplotlib.pyplot as plt

PRETRAIN_PRESET_FOR_PLOT = "Basic"  # "Light"로 학습했다면 "Light"로 변경
history_path = Path("outputs") / f"pretrain_{PRETRAIN_PRESET_FOR_PLOT.lower()}" / "pretrain_history.json"

if not history_path.exists():
    raise FileNotFoundError(f"학습 기록 파일이 없습니다: {history_path}. 본 사전 학습 셀을 먼저 실행하세요.")

with history_path.open("r", encoding="utf-8") as f:
    history = json.load(f)

if history.get("eval_step"):
    plt.figure(figsize=(8, 5))
    plt.plot(history["eval_step"], history["eval_train_loss"], label="Train loss")
    plt.plot(history["eval_step"], history["eval_val_loss"], label="Validation loss")
    plt.xlabel("Global step")
    plt.ylabel("Loss")
    plt.title(f"{PRETRAIN_PRESET_FOR_PLOT} Pretraining Loss by Step")
    plt.legend()
    plt.grid(True)
    plt.show()

if history.get("train_loss"):
    epochs = list(range(1, len(history["train_loss"]) + 1))
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], marker="o", label="Train loss")
    plt.plot(epochs, history["val_loss"], marker="o", label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{PRETRAIN_PRESET_FOR_PLOT} Pretraining Loss by Epoch")
    plt.legend()
    plt.grid(True)
    plt.show()

if history.get("epoch_time_sec"):
    epochs = list(range(1, len(history["epoch_time_sec"]) + 1))
    plt.figure(figsize=(8, 4))
    plt.bar(epochs, history["epoch_time_sec"])
    plt.xlabel("Epoch")
    plt.ylabel("Seconds")
    plt.title(f"{PRETRAIN_PRESET_FOR_PLOT} Epoch Training Time")
    plt.grid(axis="y")
    plt.show()

print("history_path:", history_path)
print("final_checkpoint_path:", history.get("final_checkpoint_path"))
print("total_training_time_sec:", history.get("total_training_time_sec"))


## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

In [ ]:
from pathlib import Path
import json
import torch
from torch.utils.data import DataLoader

from model import GPTModel
from train import load_checkpoint
from finetune import (
    ReviewSentimentDataset,
    GPTForSequenceClassification,
    train_epoch_sentiment,
    evaluate_sentiment,
)

def read_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_data = read_jsonl("data/nsmc_sentiment_train.jsonl")
val_data = read_jsonl("data/nsmc_sentiment_val.jsonl")
test_data = read_jsonl("data/nsmc_sentiment_test.jsonl")

# 처음 확인할 때는 작게, 실제 제출용은 None으로 두세요.
MAX_TRAIN_SAMPLES = 5000
MAX_VAL_SAMPLES = 1000
MAX_TEST_SAMPLES = 1000

if MAX_TRAIN_SAMPLES:
    train_data = train_data[:MAX_TRAIN_SAMPLES]
if MAX_VAL_SAMPLES:
    val_data = val_data[:MAX_VAL_SAMPLES]
if MAX_TEST_SAMPLES:
    test_data = test_data[:MAX_TEST_SAMPLES]

train_loader = DataLoader(
    ReviewSentimentDataset(train_data, tokenizer, max_length=config["context_length"]),
    batch_size=16,
    shuffle=True,
)

val_loader = DataLoader(
    ReviewSentimentDataset(val_data, tokenizer, max_length=config["context_length"]),
    batch_size=16,
    shuffle=False,
)

test_loader = DataLoader(
    ReviewSentimentDataset(test_data, tokenizer, max_length=config["context_length"]),
    batch_size=16,
    shuffle=False,
)

backbone = GPTModel(config).to(device)
load_checkpoint(
    model=backbone,
    optimizer=None,
    path=str(output_dir / "final_checkpoint.pt"),
    device=device,
)

clf_model = GPTForSequenceClassification(backbone, num_labels=2, drop_rate=0.1).to(device)
optimizer = torch.optim.AdamW(clf_model.parameters(), lr=1e-4)

best_val_loss = float("inf")
best_path = output_dir / "best_sentiment_model.pt"

finetune_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

for epoch in range(20):
    train_loss, train_acc = train_epoch_sentiment(
        clf_model, train_loader, optimizer, device
    )
    val_loss, val_acc = evaluate_sentiment(
        clf_model, val_loader, device
    )

    finetune_history["train_loss"].append(train_loss)
    finetune_history["train_acc"].append(train_acc)
    finetune_history["val_loss"].append(val_loss)
    finetune_history["val_acc"].append(val_acc)

    print(
        f"epoch {epoch + 1}: "
        f"train loss {train_loss:.4f}, train acc {train_acc:.4f}, "
        f"val loss {val_loss:.4f}, val acc {val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(clf_model.state_dict(), best_path)

clf_model.load_state_dict(torch.load(best_path, map_location=device))

test_loss, test_acc = evaluate_sentiment(
    clf_model, test_loader, device
)

finetune_history["test_loss"] = test_loss
finetune_history["test_acc"] = test_acc

with open(output_dir / "finetune_history.json", "w", encoding="utf-8") as f:
    json.dump(finetune_history, f, ensure_ascii=False, indent=2)

print("test loss:", test_loss)
print("test accuracy:", test_acc)

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, len(finetune_history["train_loss"]) + 1))

plt.figure(figsize=(8, 5))
plt.plot(epochs, finetune_history["train_loss"], marker="o", label="Train loss")
plt.plot(epochs, finetune_history["val_loss"], marker="o", label="Validation loss")
plt.axhline(
    finetune_history["test_loss"],
    linestyle="--",
    label=f"Test loss: {finetune_history['test_loss']:.4f}",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Sentiment Fine-tuning Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, finetune_history["train_acc"], marker="o", label="Train accuracy")
plt.plot(epochs, finetune_history["val_acc"], marker="o", label="Validation accuracy")
plt.axhline(
    finetune_history["test_acc"],
    linestyle="--",
    label=f"Test accuracy: {finetune_history['test_acc']:.4f}",
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Sentiment Fine-tuning Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [ ]:
run_pytest("tests/")